In [4]:
"""
Modèle de détection de fraude (VENTES) — Isolation Forest
===========================================================
SmartRent Platform - PFE Data Engineering

Approche non-supervisée : détection d'anomalies sur les annonces de vente.
Signal principal : écart entre prix affiché et prix attendu (modèle 16a).

    écart_prix_pct = (prix_affiché - prix_attendu) / prix_attendu

SORTIES
-------
- modele_fraude_vente.joblib
- rapport_fraude_ventes.md

USAGE
-----
python train_model_fraude_ventes.py
"""

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

DATA_PATH = "dataset_ventes_final.csv"
PRICE_MODEL_PATH = "modele_estimation_vente.joblib"

DISPLAY_COLUMNS = ["titre", "ville", "quartier", "prix_dh", "prix_attendu", "ecart_prix_pct"]
CONTAMINATION = 0.05  # ~5% d'annonces suspectes


def detect_price_model_features(df: pd.DataFrame) -> list:
    """
    Détecte les features attendues par le modèle d'estimation de prix.
    Exclut la cible et les colonnes texte.
    """
    exclude = {
        "prix_dh", "source", "titre", "ville", "quartier",
        "listing_id", "url", "description", "equipements",
    }
    return [c for c in df.columns if c not in exclude]


def compute_expected_price(df: pd.DataFrame, price_model, feature_cols: list) -> pd.Series:
    X = df[feature_cols].copy()
    for col in X.columns:
        if X[col].dtype == bool:
            X[col] = X[col].astype(int)
        if X[col].dtype == object:
            X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)
    return pd.Series(price_model.predict(X), index=df.index)


def make_synthetic_fraud(df: pd.DataFrame, n: int = 40, seed: int = 42) -> pd.DataFrame:
    """Clone n annonces réelles en divisant leur prix par 3 à 7 — simule des
    annonces-appâts manifestement sous le marché."""
    rng = np.random.default_rng(seed)
    sample = df.sample(n, random_state=seed).copy()
    factors = rng.uniform(0.15, 0.35, size=n)  # prix à 15-35% du prix réel
    sample["prix_dh"] = (sample["prix_dh"] * factors).round(-2)
    sample["titre"] = "[SYNTHETIQUE] " + sample["titre"].astype(str)
    return sample


def main():
    print("=" * 70)
    print("Detection de fraude — Annonces de VENTE")
    print("=" * 70)

    df = pd.read_csv(DATA_PATH)
    price_model = joblib.load(PRICE_MODEL_PATH)

    # Détection automatique des features du modèle de prix
    price_model_features = detect_price_model_features(df)
    print("\nFeatures du modele de prix detectees ({}): {}".format(
        len(price_model_features), price_model_features))

    # Calcul du prix attendu et de l'écart
    df["prix_attendu"] = compute_expected_price(df, price_model, price_model_features)
    df["ecart_prix_pct"] = (df["prix_dh"] - df["prix_attendu"]) / df["prix_attendu"]

    print("\nCatalogue : {} biens".format(len(df)))
    print("Ecart prix moyen : {:.1%} | ecart-type : {:.1%}".format(
        df["ecart_prix_pct"].mean(), df["ecart_prix_pct"].std()))

    # --- Features pour l'Isolation Forest ---
    # On prend toutes les features numériques (sauf prix) + écart_prix_pct
    # Les flags *_imputed sont inclus car une annonce avec beaucoup de données
    # manquantes est statistiquement plus suspecte
    fraud_features = [c for c in price_model_features if c != "prix_dh"] + ["ecart_prix_pct"]
    print("Features de fraude ({}): {}".format(len(fraud_features), fraud_features))

    # --- Entraînement Isolation Forest ---
    iso = IsolationForest(
        n_estimators=300,
        contamination=CONTAMINATION,
        random_state=42,
        n_jobs=-1,
    )
    iso.fit(df[fraud_features])

    df["anomaly_score"] = -iso.score_samples(df[fraud_features])
    df["suspect_isolation_forest"] = iso.predict(df[fraud_features]) == -1

    # --- Règle explicite complémentaire sur le prix ---
    # L'Isolation Forest dilue parfois le signal prix quand les autres
    # features restent normales. Une règle directe sur le percentile bas
    # de l'écart de prix comble ce point faible.
    seuil_prix_bas = df["ecart_prix_pct"].quantile(0.05)
    df["suspect_prix_bas"] = df["ecart_prix_pct"] < seuil_prix_bas

    df["est_suspect"] = df["suspect_isolation_forest"] | df["suspect_prix_bas"]

    print("\nSeuil regle prix bas (5e percentile) : {:.1%}".format(seuil_prix_bas))

    n_suspects = df["est_suspect"].sum()
    print("\n{} biens marques suspects sur {} ({:.1f}%)".format(
        n_suspects, len(df), 100 * n_suspects / len(df)))

    print("\n=== Top 10 annonces les plus suspectes (donnees reelles) ===")
    top_suspects = df.sort_values("anomaly_score", ascending=False).head(10)
    print(top_suspects[DISPLAY_COLUMNS].to_string(index=False))

    # --- Validation : injection de fraude synthétique ---
    synthetic = make_synthetic_fraud(df, n=40)
    synthetic["prix_attendu"] = compute_expected_price(synthetic, price_model, price_model_features)
    synthetic["ecart_prix_pct"] = (synthetic["prix_dh"] - synthetic["prix_attendu"]) / synthetic["prix_attendu"]

    synthetic_scores = -iso.score_samples(synthetic[fraud_features])
    synthetic_isolation_flags = iso.predict(synthetic[fraud_features]) == -1
    synthetic_prix_flags = synthetic["ecart_prix_pct"] < seuil_prix_bas
    synthetic_flags = synthetic_isolation_flags | synthetic_prix_flags
    detection_rate = synthetic_flags.mean()

    print("\n=== Validation — {} fausses annonces synthetiques (prix divise par 3-7) ===".format(len(synthetic)))
    print("Taux de detection : {:.1%} ({}/{})".format(
        detection_rate, synthetic_flags.sum(), len(synthetic)))

    missed = synthetic[~synthetic_flags][DISPLAY_COLUMNS].head(5)
    print("\nExemples non detectes (faux negatifs) :")
    if len(missed):
        print(missed.to_string(index=False))
    else:
        print("  (aucun — detection a 100%)")

    # --- Sauvegarde ---
    artifacts = {
        "model": iso,
        "features": fraud_features,
        "seuil_prix_bas": seuil_prix_bas,
        "price_model_features": price_model_features,
    }
    joblib.dump(artifacts, "modele_fraude_vente.joblib")
    print("\nModele sauvegarde : modele_fraude_vente.joblib")

    # --- Rapport ---
    report_lines = [
        "# Rapport — Detection de fraude VENTES (Isolation Forest, v1)",
        "",
        "## Approche",
        "Approche hybride : (1) regle explicite sur le percentile bas de l'ecart",
        "au prix attendu (calcule par le modele d'estimation de vente), et (2) un",
        "Isolation Forest multivarie (surface, chambres, salles de bain, quartier,",
        "ecart de prix, flags imputed) pour capter les anomalies non reductibles",
        "au seul prix.",
        "",
        "## Catalogue",
        "- {} biens analyses (fusion Mubawab + Sarouty Ventes)".format(len(df)),
        "- Features de fraude ({}): {}".format(len(fraud_features), ", ".join(fraud_features)),
        "",
        "## Resultats sur le catalogue reel",
        "- {} biens marques suspects ({:.1f}%)".format(n_suspects, 100 * n_suspects / len(df)),
        "- Ecart de prix moyen au marche : {:.1%}".format(df["ecart_prix_pct"].mean()),
        "",
        "### Top 10 les plus suspects",
        top_suspects[DISPLAY_COLUMNS].to_string(index=False),
        "",
        "**Attention** : un score eleve ne prouve pas la fraude — cela signale",
        "une annonce a verifier manuellement (prix anormalement bas pour ses",
        "caracteristiques, qui peut aussi s'expliquer par une urgence de vente,",
        "un bien a renover, une erreur de saisie, etc.). C'est un outil d'aide",
        "a la moderation, pas une decision automatique.",
        "",
        "## Validation — fraude synthetique",
        "{} annonces synthetiques creees (clones de biens reels, prix divise".format(len(synthetic)),
        "par 3 a 7 — scenario 'trop beau pour etre vrai').",
        "",
        "**Taux de detection : {:.1%}** ({}/{})".format(detection_rate, synthetic_flags.sum(), len(synthetic)),
        "",
        "## Limites connues",
        "- Aucune vraie donnee de fraude pour valider en conditions reelles — la",
        "  validation synthetique teste seulement le cas 'prix anormalement bas',",
        "  pas les autres patterns (faux compte, photos volees, usurpation d'identite...)",
        "- Le seuil de contamination (5%) est une hypothese arbitraire",
        "- Depend de la qualite du modele d'estimation de prix (R2=0.55-0.60)",
        "",
        "## Prochaine etape",
        "1. Exposer via `/predict/fraude-vente` dans le micro-service IA",
        "2. Une fois `reclamation` alimentee par de vrais signalements,",
        "   re-entrainer avec ces cas comme reference",
        "3. Ajouter des signaux comportementaux (frequence de publication,",
        "   repetitions d'annonces similaires) une fois le volume suffisant",
        "",
        "---",
        "*Genere automatiquement par train_model_fraude_ventes.py*",
    ]

    with open("rapport_fraude_ventes.md", "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    print("Rapport sauvegarde : rapport_fraude_ventes.md")
    print("=" * 70)


if __name__ == "__main__":
    main()

Detection de fraude — Annonces de VENTE

Features du modele de prix detectees (28): ['chambres', 'chambres_imputed', 'quartier_prix_estime', 'salles_de_bain', 'salles_de_bain_imputed', 'surface_m2', 'surface_m2_imputed', 'ville_Agadir', 'ville_Autre', 'ville_Bouskoura', 'ville_Casablanca', 'ville_Dar Bouazza', 'ville_El Jadida', 'ville_El Mansouria', 'ville_Fès', 'ville_Inconnu', 'ville_Kénitra', 'ville_Marrakech', 'ville_Martil', 'ville_Mohammedia', 'ville_Médiouna', 'ville_Nouaceur', 'ville_Rabat', 'ville_Salé', 'ville_Skhirat', 'ville_Tanger', 'ville_Temara', 'ville_Zenata']

Catalogue : 2470 biens
Ecart prix moyen : 1.1% | ecart-type : 47.7%
Features de fraude (29): ['chambres', 'chambres_imputed', 'quartier_prix_estime', 'salles_de_bain', 'salles_de_bain_imputed', 'surface_m2', 'surface_m2_imputed', 'ville_Agadir', 'ville_Autre', 'ville_Bouskoura', 'ville_Casablanca', 'ville_Dar Bouazza', 'ville_El Jadida', 'ville_El Mansouria', 'ville_Fès', 'ville_Inconnu', 'ville_Kénitra', 'vill